# Pruning ratio **60%** — YOLOv26m + CWD trên PASCAL VOC| | ||---|---|| Tỉ lệ prune | **60%** (L1-norm uniform, divisor 8) || Finetune | 100 epoch, CWD tau=9, kd_layers=neck, kd_warmup=5 || Batch / imgsz / seed | 16 / 640 / 0 || Tốc độ | ~13 phút/epoch (teacher forward chiếm phần lớn) || Tổng | ~20h → **2-3 phiên**, mỗi phiên tự dừng ở 10h |> Mỗi người **một ratio**. Đừng đổi `RATIO` — số của bạn đã ghi sẵn ở cell 2.## Cách chạy1. Settings → Accelerator **GPU T4 x2**, **Internet: On**2. **Add Data** → dataset **`xauduabo2`** (chứa `weights/yolo26m_baseline.pt`).   File này vừa là model để prune, vừa là teacher cho CWD — thiếu là không chạy được.3. Bấm **Save & Run All**.4. Phần Kết quả sẽ báo `CHUA` kèm số epoch đã chạy — **đó là bình thường**.   Add Data thêm output của chính lần chạy này rồi Save & Run All lại.   Lặp lại đến khi báo `CO`. Mỗi phiên tự dừng ở 10h nên output luôn được lưu.Xong thì gửi lại `results/e2e_manifest.json` trong tab Output.Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup (clone fork + cài deps)

In [ ]:
import os, subprocess, sys, pathlibWORK = pathlib.Path("/kaggle/working")REPO_DIR = WORK / "yolo"# PHAI dung fork nay, KHONG duoc "pip install ultralytics": checkpoint sau khi# prune duoc pickle voi ultralytics.nn.tasks_pruned.DetectionModelPruned - ban# Ultralytics chinh thuc khong load duoc.if not REPO_DIR.exists():    r = subprocess.run(["git", "clone", "--depth", "1", "-b", "main",                        "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], capture_output=True, text=True)    if r.returncode:        print((r.stderr or "").strip())        raise SystemExit("Clone that bai - repo phai dang o che do PUBLIC.")os.chdir(REPO_DIR)sys.path.insert(0, str(REPO_DIR))if not (REPO_DIR / "scripts" / "run_e2e.py").exists():    raise SystemExit("Thieu scripts/run_e2e.py - hay push ban moi nhat len nhanh main.")subprocess.run([sys.executable, "-m", "pip", "install", "-q",                "-r", "requirements.txt"], check=False)# Cai fork o che do editable: DDP sinh tien trinh con chay file tam ngoai repo,# neu fork khong duoc cai thi con bao "No module named 'ultralytics'".# --no-deps de khong dung toi torch/torchvision da co san.subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",                "--no-deps"], check=False)import torchimport ultralyticsprint("ultralytics", ultralytics.__version__, "| torch", torch.__version__)print("GPU:", torch.cuda.device_count(),      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Cấu hình

In [ ]:
# ==========================================================================#  TI LE PRUNING CUA BAN# ==========================================================================RATIO  = 0.6          # <<< moi nguoi mot gia tri, DUNG SUATAG    = "r60"EPOCHS = 100BATCH  = 16IMGSZ  = 640# Kaggle giet phien o 12h. Prune + CWD mat ~13 phut/epoch (do that, xem# results/yolo_compare.md) => 100 epoch khoang 20h, KHONG the xong trong mot phien.# Phai tu dung truoc 12h, neu khong lan chay bi danh dau failed va Kaggle# KHONG luu output -> khong con last.pt de resume -> mat trang ca phien.STOP_AFTER_H = 10.0   # train tu dung (mem, co final_eval)HARD_LIMIT_H = 11.0   # giet han neu dung mem khong kip#  DDP: fork da dua finetune/kd/kd_teacher/... vao ultralytics/cfg/default.yaml#  nen chung nam trong trainer.args va di duoc sang tien trinh con.#  (Truoc do chung chi la attribute cua object trainer -> con mat sach ->#   trainer.py:478 kd_enabled=False -> train 100 epoch KHONG co CWD, khong bao loi.)#  run_e2e.py con kiem tra lai default.yaml truoc khi cho chay DDP.##  Bat DDP thi phai bat cho CA NHOM: batch 16 chia 2 GPU = 8 mau/GPU, batch hieu#  dung van la 16 nhung BatchNorm chuan hoa tren 8 mau thay vi 16.USE_DDP = TrueDEVICE = "0,1" if (USE_DDP and torch.cuda.device_count() > 1) else "0"print("device  :", DEVICE, "(DDP)" if "," in DEVICE else "(1 GPU)")# Trong so baseline (yolo26m da train tren VOC) - vua la nguon de prune,# vua la teacher cho CWD.BASELINE_PATH = "/kaggle/input/datasets/xauduabo2/weights/yolo26m_baseline.pt"BASELINE = BASELINE_PATH if pathlib.Path(BASELINE_PATH).exists() else Noneif BASELINE is None:    # Du phong: Kaggle doi ten mount thi do trong /kaggle/input    for pat in ["kaggle/input/**/yolo26m_baseline.pt", "kaggle/input/**/*baseline*.pt"]:        hit = sorted(pathlib.Path("/").glob(pat))        if hit:            BASELINE = str(hit[0])            print("(khong thay duong dan mac dinh, dung:", BASELINE, ")")            breakprint("ratio   :", RATIO, "->", TAG)print("baseline:", BASELINE or "!! CHUA CO - phai Add Data truoc khi chay cell sau")

## 3. Resume

In [ ]:
# Lan chay dau se bao "Copy 0" - binh thuong.# Tu lan 2: Add Data -> Your Work -> chon output lan truoc, notebook tu chep# runs/ + weights/ + manifest ve roi train tiep tu epoch dang do.import glob, json as _jn = 0# Manifest: GOP chu khong ghi de. Neu Add Data nhieu output (vd ca cua nguoi khac)# ma ghi de thi manifest cuoi se xoa mat tien do cua chinh minh.merged = {}mf = REPO_DIR / "results" / "e2e_manifest.json"if mf.exists():    merged.update(_j.loads(mf.read_text()))for depth in range(1, 6):    for src in glob.glob("/kaggle/input/" + "*/" * depth + "yolo/results/e2e_manifest.json"):        merged.update(_j.loads(pathlib.Path(src).read_text())); n += 1if merged:    mf.parent.mkdir(parents=True, exist_ok=True)    mf.write_text(_j.dumps(merged, indent=2, ensure_ascii=False))# Neu gan NHIEU output (vd ca phien 1 lan phien 2) thi phai chon ban co NHIEU# epoch nhat. Lay "cai dau tien tim thay" la sai: thu tu glob khong xac dinh,# co the chep nham ban cu va mat vai gio train ma khong he biet.def _epochs(d):    f = pathlib.Path(d) / "results.csv"    if not f.exists():        return -1    try:        rows = [r for r in f.read_text().strip().splitlines()[1:] if r.strip()]        return int(float(rows[-1].split(",")[0])) if rows else 0    except Exception:        return 0cands = {}for depth in range(1, 6):    for src in glob.glob("/kaggle/input/" + "*/" * depth + "yolo/runs/e2e/*"):        cands.setdefault(pathlib.Path(src).name, []).append(src)for name, srcs in sorted(cands.items()):    best = max(srcs, key=_epochs)    if len(srcs) > 1:        print(f"   {name}: co {len(srcs)} ban, chon ban {_epochs(best)} epoch")    dst = REPO_DIR / "runs" / "e2e" / name    if not dst.exists():        shutil.copytree(best, dst); n += 1        print(f"   {name}: {_epochs(best)} epoch da train")for depth in range(1, 6):    for src in glob.glob("/kaggle/input/" + "*/" * depth + "yolo/weights/*.pt"):        dst = REPO_DIR / "weights" / pathlib.Path(src).name        dst.parent.mkdir(parents=True, exist_ok=True)        if not dst.exists():            shutil.copy2(src, dst); n += 1print("Copy", n, "muc tu lan chay truoc")if mf.exists():    for k, v in _j.loads(mf.read_text()).items():        print(f"   {k:<18} {v.get('weights', v.get('log', ''))}")

## 4. Prune → Finetune + CWD → Val

In [ ]:
assert BASELINE, "Chua co trong so baseline - Add Data roi chay lai cell config"cmd = [sys.executable, "scripts/run_e2e.py",       "--stage", "prune", "finetune", "val",       "--prune-ratio", str(RATIO), "--tag", TAG,       "--baseline-weights", BASELINE,       "--teacher-weights", BASELINE,       "--epochs", str(EPOCHS), "--batch", str(BATCH), "--imgsz", str(IMGSZ),       "--device", DEVICE, "--data", "VOC.yaml", "--resume",       "--stop-after-h", str(STOP_AFTER_H)]print(" ".join(cmd), flush=True)# Hai lop chan thoi gian:#   1. --stop-after-h: train tu dung sau N gio, chay final_eval, thoat GON.#   2. HARD_LIMIT_H duoi day: neu (1) khong kip (epoch dang chay qua lau, hoac treo)#      thi giet han tien trinh. Ultralytics luu last.pt moi epoch nen cung lam mat#      dung mot epoch.# Muc dich chung: notebook phai KET THUC BINH THUONG truoc moc 12h cua Kaggle.# Neu de Kaggle giet phien thi lan chay bi danh dau failed va KHONG luu output,# tuc la khong con last.pt de resume - mat trang ca phien.import signal, time as _tt0 = _t.time()proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), start_new_session=True)try:    proc.wait(timeout=HARD_LIMIT_H * 3600)except subprocess.TimeoutExpired:    print(f"\n!! Qua {HARD_LIMIT_H}h -> dung tien trinh de output kip luu.", flush=True)    # Phai giet ca NHOM tien trinh: DDP con nam trong nhom do, giet moi tien trinh    # cha se de lai con mo coi van giu GPU.    def _kill(sig):        try:            os.killpg(os.getpgid(proc.pid), sig)            return True        except Exception:            return False    if not _kill(signal.SIGTERM):        proc.terminate()    try:        proc.wait(timeout=180)    except subprocess.TimeoutExpired:        if not _kill(signal.SIGKILL):            proc.kill()        proc.wait(timeout=60)print(f"\nTien trinh ket thuc sau {(_t.time() - t0) / 3600:.2f}h")

## 5. Kết quả

In [ ]:
import jsonmf = REPO_DIR / "results" / "e2e_manifest.json"man = json.loads(mf.read_text()) if mf.exists() else {}done = "finetune@" + TAG in man and "val@" + TAG in manprint("Ti le :", RATIO)print("Xong  :", "CO" if done else "CHUA - chay lai notebook (nho Add Data output lan nay)")row = man.get("val@" + TAG, {}).get("pruned")if row:    print()    print(f"  AP50     {row['AP50']}")    print(f"  AP50-95  {row['AP50_95']}")    print(f"  Params   {row['params_M']} M")    print()    print("Gui lai cho nguoi gom ket qua:")    print("   /kaggle/working/yolo/results/e2e_manifest.json")    print("   ", man["finetune@" + TAG]["weights"])else:    csv = REPO_DIR / "runs" / "e2e" / ("finetune_" + TAG) / "results.csv"    if csv.exists():        import pandas as pd        df = pd.read_csv(csv); df.columns = df.columns.str.strip()        print(f"\nTien do: epoch {int(df['epoch'].max())}/{EPOCHS}")